# NIO Stock Analysis

- Company: NIO Inc.
- Stock ticker: NIO
- Requested analysis period: January 1, 2026–October 12, 2026
- Observed trading days: January 2, 2026–October 9, 2026 (194 records)
- Data source: Yahoo Finance through yfinance
- Prices: Adjusted closing prices in USD

## Research Question

How did NIO's stock price and daily return volatility change during
the analysis period, and what business events may explain a
significant stock price movement?

In [1]:
%pip install pandas yfinance matplotlib

In [2]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

print("Tools imported successfully.")

Tools imported successfully.


In [3]:
TICKER = "NIO"
START_DATE = "2026-01-01"
END_DATE = "2026-10-12"

nio = yf.download(
    TICKER,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=False,
)

if nio.empty:
    raise ValueError("No data downloaded. Check the connection and settings.")

nio.head()

Price,Close,High,Low,Open,Volume
Ticker,NIO,NIO,NIO,NIO,NIO
Date,,,,,
2026-01-02,5.14,5.38,5.08,5.32,40391000
2026-01-05,4.86,5.01,4.85,5.01,74650200
2026-01-06,4.78,5.04,4.74,4.96,61127400
2026-01-07,4.79,4.84,4.76,4.81,34014400
2026-01-08,4.73,4.85,4.67,4.85,33792600


In [4]:
if isinstance(nio.columns, pd.MultiIndex):
    nio = nio.xs(TICKER, axis=1, level="Ticker")

nio = nio.sort_index()
nio.head()

Price,Close,High,Low,Open,Volume
Date,,,,,
2026-01-02,5.14,5.38,5.08,5.32,40391000
2026-01-05,4.86,5.01,4.85,5.01,74650200
2026-01-06,4.78,5.04,4.74,4.96,61127400
2026-01-07,4.79,4.84,4.76,4.81,34014400
2026-01-08,4.73,4.85,4.67,4.85,33792600


In [5]:
print("First trading date:", nio.index.min().date())
print("Last trading date:", nio.index.max().date())
print("Number of rows:", len(nio))

display(nio.tail())

First trading date: 2026-01-02
Last trading date: 2026-10-09
Number of rows: 194


Price,Close,High,Low,Open,Volume
Date,,,,,
2026-10-05,3.42,3.47,3.40,3.42,26310800
2026-10-06,3.48,3.52,3.40,3.42,30394900
2026-10-07,3.54,3.55,3.43,3.45,25766400
2026-10-08,3.41,3.49,3.36,3.49,39765400
2026-10-09,3.58,3.63,3.48,3.49,44006200


In [6]:
print("Columns:", nio.columns.tolist())

print("\nData types:")
print(nio.dtypes)

print("\nMissing values:")
print(nio.isna().sum())

print("\nDuplicate dates:", nio.index.duplicated().sum())

Columns: ['Close', 'High', 'Low', 'Open', 'Volume']

Data types:
Price
Close     float64
High      float64
Low       float64
Open      float64
Volume      int64
dtype: object

Missing values:
Price
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

Duplicate dates: 0


## Data Inspection

The downloaded NIO dataset contains 194 daily records, covering
January 2, 2026 to October 9, 2026. Each row represents one trading day.

The columns are Close, High, Low, Open, and Volume. Price columns
are stored as floating-point numbers, while Volume is stored as
integers. No missing values or duplicate dates were found in the
downloaded data.

The dataset includes trading days only. Prices are adjusted because
auto_adjust=True was used during the download.

## Daily Returns

Daily return measures the percentage change in adjusted closing
price from the previous trading day.

Daily return (%) = (Current close / Previous close - 1) × 100

The first return is missing because the downloaded dataset has
no previous trading day's closing price for that observation.

In [7]:
nio["Daily_Return_Pct"] = (
    nio["Close"].pct_change(fill_method=None) * 100
)

display(nio[["Close", "Daily_Return_Pct"]].head())

Price,Close,Daily_Return_Pct
Date,,
2026-01-02,5.14,NaN
2026-01-05,4.86,-5.447466
2026-01-06,4.78,-1.646089
2026-01-07,4.79,0.209200
2026-01-08,4.73,-1.252608


Returns are calculated using all observations in the analysis
period. The first five rows below are shown only to check the calculation.

In [8]:
nio_stats = nio[["Close", "Daily_Return_Pct"]].agg(
    ["count", "mean", "median", "min", "max", "std"]
)

nio_stats.columns = [
    "Adjusted Close (USD)",
    "Daily Return (%)"
]

display(nio_stats.round(2))

,Adjusted Close (USD),Daily Return (%)
count,194.00,193.00
mean,5.01,-0.14
median,4.90,-0.38
min,3.37,-7.81
max,6.87,15.38
std,0.83,3.24


## Descriptive Statistics

NIO has 194 adjusted closing-price observations and 193 valid daily
returns. The first daily return is missing because no previous
closing price is available within the downloaded dataset.

The mean adjusted closing price was $5.01, and the median was $4.90.
Adjusted closing prices ranged from $3.37 to $6.87, with a standard
deviation of $0.83.

Daily returns ranged from -7.81% to 15.38%. The mean daily return
was approximately -0.14%, while the median was -0.38%. The standard
deviation was 3.24 percentage points, describing the spread of
daily returns during the analysis period.

The mean daily return is an arithmetic average, not the total
return over the analysis period.

In [9]:
nio_monthly = nio["Close"].groupby(
    nio.index.to_period("M")
).agg(["count", "mean"])

nio_monthly.columns = [
    "Trading Days",
    "Average Adjusted Close (USD)"
]

display(nio_monthly.round(2))

,Trading Days,Average Adjusted Close (USD)
Date,,
2026-01,20,4.72
2026-02,19,4.93
2026-03,22,5.46
2026-04,21,6.40
2026-05,20,5.84
2026-06,21,5.24
2026-07,22,4.82
2026-08,21,4.55
2026-09,21,3.67


## Monthly Average Prices

NIO's average adjusted closing price increased from $4.72 in
January to $6.40 in April, the highest monthly average among
the complete months analyzed.

After April, the monthly average declined each month, reaching
$3.67 in September, the lowest average among the complete months.

October's average was $3.46, based on only seven trading days
through October 9. It represents a partial month.

These figures describe monthly average price levels, not monthly
returns, and do not explain the causes of the price changes.